# Ines — DeepLabV3+ architecture and transfer learning

Non-authoritative learning client. Implementation lives in `dl/`.
Open from the repository root or `dl/notebooks/`. Never train on RoadAnomaly.

## 1. What you are building
The encoder extracts visual features. ASPP combines context at dilation rates 6, 12 and 18.
The decoder combines those features with stride-4 details to reconstruct boundaries.
11 CamVid classes are normal semantic categories, not anomaly categories.
Input: RGB [0,255], preprocessing inside the model. Output: semantic logits.


In [ ]:
import sys
from pathlib import Path

root = Path.cwd()
while not (root / "AGENTS.md").exists() and root != root.parent:
    root = root.parent
assert (root / "AGENTS.md").exists(), "Extract the complete project first"
sys.path.insert(0, str(root))

In [ ]:
import tensorflow as tf

from dl.models.deeplabv3plus import build, predict
from dl.training.ines_camvid import load_manifest

## 2. Architecture inspection (random weights, no training)
This section checks tensor shapes only. Random scores are not useful anomaly detections.
Use ImageNet weights only when starting the real CamVid learning experiment.


In [ ]:
config = {
    "height": 64,
    "width": 96,
    "classes": 11,
    "weights": None,
    "freeze_encoder": True,
    "decoder_channels": 16,
}
model = build(config)
model.summary()

In [ ]:
batch = tf.zeros((1, 64, 96, 3), dtype=tf.float32)
logits = model(batch, training=False)
scores = predict(model, batch)
print("Logits:", logits.shape, "Score map:", scores.shape)
assert tuple(logits.shape) == (1, 64, 96, 11)
assert tuple(scores.shape) == (1, 64, 96)

## 3. Read the implementation
At the oral defence, explain why `aspp_rate_6`, `aspp_rate_12`, `aspp_rate_18` and
`decoder_concat` exist. Explain why MSP is an uncertainty heuristic, not a calibrated
probability that an obstacle exists. ImageNet pretraining alone does not learn road classes.


In [ ]:
import inspect

print(inspect.getsource(build))

## 4. Prepare real data
Run from a terminal at the repository root:

```bash
python -m dl.data.prepare_camvid
```

The downloader pins the authors' release, verifies annotations and writes content hashes.
Whole video sequences stay in one partition. The published tutorial split is not reused,
because nearby frames must not cross the train/validation boundary.
The normality of the corpus still requires visual review before an official experiment.


In [ ]:


manifest_path = root / "configs/datasets/camvid.json"
if not manifest_path.exists():
    raise FileNotFoundError("Run python -m dl.data.prepare_camvid from the repository root")
manifest = load_manifest(root, manifest_path)
print("Classes:", manifest["classes"])
print(
    {
        part: sum(row["split"] == part for row in manifest["frames"])
        for part in ("train", "val", "test")
    }
)

## 5. Start the learning experiment deliberately
From the repository root:

```bash
python -m dl.training.ines_camvid --epochs 3
```

The encoder is frozen; the ASPP and decoder learn CamVid labels. Void pixels have zero
sample weight. Validation loss selects the checkpoint on CamVid only. Start short, inspect
predictions and losses, then coordinate partial fine-tuning with Emna.
No training runs automatically from this notebook. No fake results are included.

## 6. What remains
- Validate normal-road corpus suitability with the team, then pin an official config.
- Add augmentation and controlled fine-tuning experiments with Emna.
- Implement shared metrics once in `dl/evaluation/`, not here.
- Fix the threshold using normal-road validation, then evaluate all 60 benchmark frames.
- Add Seg-Grad-CAM over decoder/ASPP and test explanation fidelity.
- Integrate registry manifest and official experiment record.

## References
- https://keras.io/examples/vision/deeplabv3_plus/
- https://github.com/alexgkendall/SegNet-Tutorial
- ../README.md and ../../docs/adr/0010-model-lineup-and-evaluation-contract.md


## XAI implementation available
`dl.evaluation.xai.seg_gradcam` explains a fixed region of the MSP anomaly score.
Use it with the trained checkpoint and a fixed region. Compare `decoder_2`,
`aspp_rate_6`, `aspp_rate_12`, `aspp_rate_18`. Do not interpret random-weight maps.
Faithfulness deletion/insertion tests remain to be implemented before final XAI reporting.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tensorflow import keras

trained_model = keras.models.load_model(
    root / "artifacts/ines-deeplabv3plus/best.keras",
    compile=False,
)

validation_examples = [
    row for row in manifest["frames"]
    if row["split"] == "val"
][:3]

height, width = trained_model.input_shape[1:3]
cmap = plt.get_cmap("tab20", 12)

fig, axes = plt.subplots(3, 3, figsize=(15, 10))

for i, row in enumerate(validation_examples):
    with Image.open(root / row["image"]) as image:
        rgb = np.asarray(
            image.convert("RGB").resize(
                (width, height), Image.Resampling.BILINEAR
            )
        )

    with Image.open(root / row["mask"]) as mask:
        ground_truth = np.asarray(
            mask.resize((width, height), Image.Resampling.NEAREST)
        )

    logits = trained_model(
        rgb[None].astype(np.float32), training=False
    ).numpy()[0]
    prediction = np.argmax(logits, axis=-1)

    # Les pixels void (11) sont masqués dans les deux visualisations.
    ignored = ground_truth == manifest["ignore_label"]

    axes[i, 0].imshow(rgb)
    axes[i, 0].set_title("Image de validation")

    axes[i, 1].imshow(
        np.ma.array(ground_truth, mask=ignored),
        cmap=cmap, vmin=0, vmax=11,
        interpolation="nearest",
    )
    axes[i, 1].set_title("Annotation réelle")

    axes[i, 2].imshow(
        np.ma.array(prediction, mask=ignored),
        cmap=cmap, vmin=0, vmax=11,
        interpolation="nearest",
    )
    axes[i, 2].set_title("Prédiction après 1 epoch")

    for ax in axes[i]:
        ax.axis("off")

plt.tight_layout()
plt.show()

print("Classes :", dict(enumerate(manifest["classes"])))